# gradprobe: gradient-conflict probe (temporary)

Measures, on the final checkpoint of a finished run, how the loss terms of the two teachers (cosine, similarity, ranking) push the shared encoder: the cosine between the gradients of every pair of terms (negative = conflict). Needs a GPU (an L4 is enough), the run's folder on Drive and the 200K mix. Nothing is trained or saved except the JSON with the averaged numbers. Delete this notebook when the question is answered.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
import glob, importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')
os.chdir(REPO_DIR)
if importlib.util.find_spec('turkish_distill') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()

# Run to probe: the plain BGE + 8K pair at lr 1e-4, 5400 steps (67.16). Its config.yaml in the run folder holds the exact settings.
RUN_DIR = f'{DRIVE_ROOT}/runs/scale_two_lr1e4_mix200k_s5400/20261027-063854-79eb81a'
BATCHES = 10          # batches averaged
PARAMS = 'all'        # or 'last:4' if the GPU runs out of memory
OUT = f'{DRIVE_ROOT}/eval/gradprobe_pair_lr1e4.json'

assert os.path.isdir(f'{RUN_DIR}/checkpoints'), f'no checkpoints in {RUN_DIR}'
print(sorted(os.listdir(f'{RUN_DIR}/checkpoints')))
get_ipython().system(f'python -m turkish_distill.grad_conflict --config {RUN_DIR}/config.yaml --run-dir {RUN_DIR} --batches {BATCHES} --params {PARAMS} --out {OUT}')


In [ ]:
from google.colab import runtime
runtime.unassign()